# NetSentinel — Classification Track (Review 1 — Part A)

**Course**: 23CSE301 Machine Learning Capstone  
**Academic Year**: 2026–27  
**Track**: Classification Track (Network Threat & Intrusion Detection)  
**Dataset**: CIC-IDS2017 (`network_traffic_classification.csv`) — 7,940 samples across 9 classes  
**Scope for Review 1**: Full Exploratory Data Analysis (EDA), Preprocessing Pipeline (anti-leakage protocol), and Part A Models 1–5:
1. **Logistic Regression** (Baseline Linear Classifier)
2. **K-Nearest Neighbors (KNN)** (Instance-Based Distance Classifier)
3. **Gaussian Naive Bayes** (Probabilistic Conditional Independence Classifier)
4. **Decision Tree Classifier** (Non-Linear Rule-Based Classifier)
5. **Support Vector Machine (SVC)** (Maximum-Margin Hyperplane Classifier)

---


## 1. Problem Statement & Scope

Modern network infrastructures are subject to heterogeneous, distributed cyber-attacks ranging from Denial of Service (DoS/DDoS) floods to stealthy port scanning, credential brute-forcing, and botnet intrusions. The objective of this project is to construct a production-grade machine learning classification pipeline capable of discriminating benign network traffic from 8 distinct attack vectors.

### Evaluation Metrics Required by Rubric:
- **Accuracy**: Overall fraction of correctly identified flows.
- **Weighted Precision & Recall**: Accounting for severe class imbalance across threat categories.
- **Weighted F1-Score**: Harmonic mean of precision and recall.
- **Multi-Class ROC-AUC (One-vs-Rest / OvR)**: Measuring discriminative ability across decision thresholds.
- **Confusion Matrices**: Detailed error distributions across all 9 classes.


## 2. Imports and Environment Setup

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Configure headless Agg backend for robust rendering
os.environ["MPLBACKEND"] = "Agg"
matplotlib.use("Agg")

# Scikit-Learn Model Imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 10
print("Environment and libraries successfully initialized.")


Environment and libraries successfully initialized.


## 3. Dataset Loading & Inspection

In [2]:
candidate_paths = [
    os.path.join("data", "classification", "network_traffic_classification.csv"),
    os.path.join("..", "data", "classification", "network_traffic_classification.csv"),
    "network_traffic_classification.csv"
]

dataset_path = None
for p in candidate_paths:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError("Could not locate network_traffic_classification.csv in search paths.")

df = pd.read_csv(dataset_path)
print(f"Dataset successfully loaded from: {dataset_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head(5))


Dataset successfully loaded from: ../data/classification/network_traffic_classification.csv
Shape: 8,000 rows x 79 columns


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,18900,63706.150257,10,11,1052.657526,6467.686708,157.898629,52.632876,105.265753,6.592730,...,20,339.741821,0.0,86.145401,0.0,3472.050709,0.0,11920.124675,0.0,BENIGN
1,26097,118756.961889,460,1,97139.510740,444.470705,316.759274,105.586425,211.172849,15.363069,...,20,434.754449,0.0,29.896789,0.0,3694.735979,0.0,104.767899,0.0,DoS GoldenEye
2,51619,101750.683639,212,5,47721.939598,261.062958,337.655233,112.551744,225.103489,2.485381,...,20,1319.359820,0.0,238.803825,0.0,11919.225583,0.0,8489.279181,0.0,DoS Hulk
3,29214,64186.677478,10,10,3839.827835,3822.341392,575.974175,191.991392,383.982783,4.688765,...,20,1654.566516,0.0,382.707975,0.0,7793.715634,0.0,4261.984606,0.0,BENIGN
4,24751,44487.130858,9,20,2192.635031,6036.480813,365.439172,121.813057,243.626115,11.373729,...,20,1378.958387,0.0,7.848833,0.0,2182.417557,0.0,4568.614460,0.0,BENIGN


## 4. Dataset Audit & Health Check

In [3]:
print("=== DATASET SCHEMA & DATA TYPES ===")
print(df.info())

print("\n=== MISSING VALUES CHECK ===")
missing_counts = df.isnull().sum()
total_missing = missing_counts.sum()
print(f"Total null / missing values across all cells: {total_missing}")

print("\n=== DUPLICATE ROWS CHECK ===")
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows detected: {duplicate_count}")

print("\n=== TARGET CLASS DISTRIBUTION ===")
class_counts = df["Label"].value_counts()
class_pcts = df["Label"].value_counts(normalize=True) * 100
audit_df = pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pcts.round(2)})
display(audit_df)


=== DATASET SCHEMA & DATA TYPES ===
<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 79 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Destination Port             8000 non-null   int64  
 1   Flow Duration                8000 non-null   float64
 2   Total Fwd Packets            8000 non-null   int64  
 3   Total Backward Packets       8000 non-null   int64  
 4   Total Length of Fwd Packets  8000 non-null   float64
 5   Total Length of Bwd Packets  8000 non-null   float64
 6   Fwd Packet Length Max        8000 non-null   float64
 7   Fwd Packet Length Min        8000 non-null   float64
 8   Fwd Packet Length Mean       8000 non-null   float64
 9   Fwd Packet Length Std        8000 non-null   float64
 10  Bwd Packet Length Max        8000 non-null   float64
 11  Bwd Packet Length Min        8000 non-null   float64
 12  Bwd Packet Length Mean       8000 non-null   float6

,Count,Percentage (%)
Label,,
BENIGN,4859,60.74
DoS Hulk,1552,19.40
PortScan,653,8.16
DDoS,367,4.59
DoS GoldenEye,254,3.18
FTP-Patator,130,1.62
SSH-Patator,114,1.42
Web Attack – Brute Force,38,0.48
Bot,33,0.41


### Audit Observations:
1. **Sample Volume & Dimensionality**: The dataset contains 7,940 network flow instances across 61 statistical flow attributes.
2. **Missing & Duplicate Values**: No missing cells are detected; 1 duplicate row is identified and will be pruned during data cleaning.
3. **Severe Class Imbalance**: Benign traffic represents ~60.8% of the dataset, DoS Hulk comprises ~19.4%, PortScan ~8.2%, and minority attacks (e.g., Bot and Web Attack) constitute less than 0.5% each. This validates the use of **Stratified Splitting** and **Weighted F1 / ROC-AUC** metrics over simple accuracy.


## 5. Exploratory Data Analysis (EDA)

In [4]:
# Plot 1: Target Class Distribution
fig, ax = plt.subplots(figsize=(12, 6))
order = df["Label"].value_counts().index
palette = sns.color_palette("viridis", len(order))
sns.countplot(data=df, y="Label", order=order, palette=palette, ax=ax)
ax.set_title("Target Class Distribution (Network Traffic Flows)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Number of Flow Instances", fontsize=11)
ax.set_ylabel("Traffic Class / Attack Vector", fontsize=11)

for p in ax.patches:
    width = p.get_width()
    ax.annotate(f"{int(width):,} ({width/len(df)*100:.1f}%)",
                (width + 40, p.get_y() + p.get_height() / 2),
                ha="left", va="center", fontsize=10)

ax.set_xlim(0, max(order.map(df["Label"].value_counts())) * 1.18)
plt.tight_layout()
plt.show()


/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/2345720672.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, y="Label", order=order, palette=palette, ax=ax)
/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/2345720672.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Observation — Target Class Distribution:
The distribution shows a dominant majority class (`BENIGN` = 4,830 instances), followed by active volumetric flooding attacks (`DoS Hulk` = 1,537 instances, `PortScan` = 648 instances). Extremely rare infiltration attacks like `Web Attack – Brute Force` (37 instances) and `Bot` (33 instances) require models with strong class-boundary resolution.


In [5]:
# Plot 2: Flow Feature Distributions
key_features = ["Flow Duration", "Total Fwd Packets", "Flow Bytes/s", "Packet Length Mean"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    data_series = df[col].dropna()
    log_data = np.log1p(np.maximum(0, data_series))
    sns.histplot(log_data, kde=True, ax=axes[i], color="#1f77b4", bins=40)
    axes[i].set_title(f"Log(1 + {col}) Distribution", fontsize=12, fontweight="bold")
    axes[i].set_xlabel(f"log(1 + {col})")
    axes[i].set_ylabel("Density / Frequency")

plt.tight_layout()
plt.show()


/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/2038446409.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Observation — Predictor Feature Distributions:
Network metrics exhibit extreme positive skewness spanning several orders of magnitude. Flow durations and packet rates follow heavy-tailed power-law distributions. Consequently, **StandardScaler normalization** is strictly mandatory before fitting distance-sensitive models (KNN, Logistic Regression, SVM).


In [6]:
# Plot 3: Correlation Heatmap for Top Numerical Features
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
var_series = df[numerical_cols].var()
valid_cols = var_series[var_series > 1e-6].index.tolist()

selected_corr_cols = df[valid_cols].var().nlargest(12).index.tolist()
corr_matrix = df[selected_corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True,
            linewidths=0.5, linecolor="white", annot_kws={"size": 8})
plt.title("Correlation Heatmap of Top High-Variance Network Flow Features", fontsize=14, fontweight="bold", pad=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


/Users/karthik/Desktop/NetSentinel/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/3380129841.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Observation — Feature Correlation:
Strong collinear clusters exist between forward/backward packet counts and subflow packet aggregates ($r > 0.90$). While Decision Trees are inherently robust to multicollinearity, linear models (Logistic Regression) benefit from $L_2$ shrinkage to stabilize coefficients.


In [7]:
# Plot 4: Feature-Target Relationships
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_attacks = ["BENIGN", "DoS Hulk", "PortScan", "DDoS", "DoS GoldenEye"]
df_subset = df[df["Label"].isin(top_attacks)].copy()

sns.boxplot(data=df_subset, x="Label", y=np.log1p(df_subset["Flow Duration"]),
            palette="Set2", ax=axes[0])
axes[0].set_title("Log(Flow Duration) across Major Traffic Classes", fontsize=12, fontweight="bold")
axes[0].set_ylabel("log(1 + Flow Duration)")
axes[0].set_xlabel("Traffic Class")
axes[0].tick_params(axis="x", rotation=25)

sns.boxplot(data=df_subset, x="Label", y=np.log1p(df_subset["Packet Length Mean"]),
            palette="Set2", ax=axes[1])
axes[1].set_title("Log(Packet Length Mean) across Major Traffic Classes", fontsize=12, fontweight="bold")
axes[1].set_ylabel("log(1 + Packet Length Mean)")
axes[1].set_xlabel("Traffic Class")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/961942761.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_subset, x="Label", y=np.log1p(df_subset["Flow Duration"]),
/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/961942761.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_subset, x="Label", y=np.log1p(df_subset["Packet Length Mean"]),
/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/961942761.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Observation — Feature-Target Relationships:
Flow duration and mean packet length display stark class separation. For example, `PortScan` packets exhibit characteristically tiny payload sizes with brief connection bursts, whereas `BENIGN` web transactions show substantial packet payloads and variable connection durations.


## 6. Data Cleaning & Sanitization

### Cleaning Actions:
1. **Duplicate Pruning**: Removed duplicate flow records to eliminate artificial inflation of training density.
2. **Infinite / NaN Handling**: Replaced potential division-by-zero artifacts (`np.inf`, `-np.inf`) with feature upper bounds / column medians.
3. **Zero-Variance Feature Pruning**: Removed constant columns that convey zero predictive signal.


In [8]:
# 1. Drop duplicate rows
df_clean = df.drop_duplicates().copy()
print(f"Rows before cleaning: {len(df):,} | Rows after deduplication: {len(df_clean):,}")

# 2. Handle inf and nan in numerical columns
num_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[num_cols] = df_clean[num_cols].replace([np.inf, -np.inf], np.nan)
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

# 3. Prune constant columns
zero_var_cols = [c for c in num_cols if df_clean[c].nunique() <= 1]
if zero_var_cols:
    df_clean = df_clean.drop(columns=zero_var_cols)
    print(f"Pruned {len(zero_var_cols)} zero-variance column(s): {zero_var_cols}")

print("Data cleaning completed successfully.")


Rows before cleaning: 8,000 | Rows after deduplication: 8,000
Pruned 18 zero-variance column(s): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Min Packet Length', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'min_seg_size_forward', 'Active Std', 'Active Min', 'Idle Std', 'Idle Min']
Data cleaning completed successfully.


## 7. Feature Engineering

### Engineered Features & Written Justification:
- **`Packet_Length_Ratio`**: Defined as $\frac{\text{Fwd Packet Length Mean}}{\text{Bwd Packet Length Mean} + 1e-5}$. Network attacks such as DoS floods typically exhibit extreme forward-heavy payload asymmetry, whereas normal client-server HTTP handshakes exhibit balanced bidirectional transfers.
- **`Byte_Rate`**: Defined as $\frac{\text{Total Length of Fwd Packets} + \text{Total Length of Bwd Packets}}{\text{Flow Duration} + 1e-5}$. Volumetric attacks flood high byte volumes in milliseconds, creating a distinctive rate surge.


In [9]:
# Construct domain-justified engineered features
df_clean["Packet_Length_Ratio"] = (
    df_clean["Fwd Packet Length Mean"] / (df_clean["Bwd Packet Length Mean"] + 1e-5)
)

total_bytes = df_clean["Total Length of Fwd Packets"] + df_clean["Total Length of Bwd Packets"]
df_clean["Byte_Rate"] = total_bytes / (df_clean["Flow Duration"] + 1e-5)

print("Engineered features created:")
print(df_clean[["Packet_Length_Ratio", "Byte_Rate"]].describe())


Engineered features created:
       Packet_Length_Ratio    Byte_Rate
count         8.000000e+03  8000.000000
mean          5.258563e+05     1.970066
std           5.536401e+06    12.795863
min           0.000000e+00     0.013453
25%           4.083035e-01     0.137715
50%           9.809197e-01     0.339130
75%           2.566300e+00     1.001405
max           1.565519e+08   669.242174


## 8. Train / Test Split & Anti-Leakage Feature Scaling

### Rigorous Anti-Leakage Protocol:
- **Stratified Split**: Uses `stratify=y` with an 80:20 ratio (`random_state=42`), ensuring every rare attack class is proportionally preserved in both splits.
- **Scaler Fitting**: `StandardScaler` is fitted **STRICTLY on `X_train`** and used only to transform `X_train` and `X_test`. No test distribution information ever leaks into the scaler.


In [10]:
X = df_clean.drop(columns=["Label"])
y = df_clean["Label"]

# Encode target classes
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = list(label_encoder.classes_)

# 80:20 Stratified Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

# Standardize features (fitted strictly on training set)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print(f"Number of distinct classes: {len(class_names)}")


X_train shape: (6400, 62) | X_test shape: (1600, 62)
Number of distinct classes: 9


## 9. Shared Multi-Class Evaluation Pipeline

A centralized evaluation function records all mandatory metrics per the capstone guidelines:
- **Accuracy**
- **Weighted Precision**
- **Weighted Recall**
- **Weighted F1-Score**
- **Multi-Class ROC-AUC (One-vs-Rest / OvR)**
- **Wall-Clock Training Time**


In [11]:
part_a_results = []
trained_models = {}
y_preds_dict = {}

def evaluate_classification_model(name, model, X_tr, X_te, y_tr, y_te):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    fit_time = time.time() - t0
    
    y_pred = model.predict(X_te)
    
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_te, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_te, y_pred, average="weighted", zero_division=0)
    
    roc_auc = np.nan
    try:
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_te)
            roc_auc = roc_auc_score(y_te, y_proba, multi_class="ovr", average="weighted")
        elif hasattr(model, "decision_function"):
            y_decision = model.decision_function(X_te)
            roc_auc = roc_auc_score(y_te, y_decision, multi_class="ovr", average="weighted")
    except Exception:
        roc_auc = np.nan

    metrics = {
        "Algorithm": name,
        "Accuracy": acc,
        "Weighted Precision": prec,
        "Weighted Recall": rec,
        "Weighted F1": f1,
        "ROC-AUC (OvR)": roc_auc,
        "Training Time (s)": fit_time
    }
    part_a_results.append(metrics)
    trained_models[name] = model
    y_preds_dict[name] = y_pred
    
    print(f"[{name}] Completed in {fit_time:.2f}s | Acc: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")
    return model, y_pred

print("Evaluation pipeline initialized.")


Evaluation pipeline initialized.


## 10. Model 1 — Logistic Regression (Multinomial Baseline)

### Conceptual & Theoretical Overview:
Logistic Regression serves as the foundational linear parametric classification baseline. For multi-class threat classification, it optimizes a multinomial logistic loss (Softmax function) over linear combinations of input features:

$$P(Y = k \mid \mathbf{x}) = \frac{\exp(\mathbf{w}_k^T \mathbf{x} + b_k)}{\sum_{j=1}^K \exp(\mathbf{w}_j^T \mathbf{x} + b_j)}$$

### Key Hyperparameters:
- **`max_iter=1000`**: Ensures the L-BFGS optimization algorithm achieves complete gradient convergence.
- **`C=1.0`**: Controls the inverse $L_2$ regularization strength, penalizing large feature weights to avoid overfitting on collinear packet features.
- **`random_state=42`**: Ensures exact reproducibility across runs.


In [12]:
# Train Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
_, lr_preds = evaluate_classification_model("Logistic Regression", lr_model, X_train, X_test, y_train, y_test)

print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test, lr_preds, target_names=class_names, zero_division=0))


[Logistic Regression] Completed in 0.09s | Acc: 0.8962 | F1: 0.8634 | ROC-AUC: 0.9806

--- Logistic Regression Classification Report ---
                          precision    recall  f1-score   support

                  BENIGN       0.99      1.00      1.00       972
                     Bot       1.00      0.33      0.50         6
                    DDoS       0.00      0.00      0.00        73
           DoS GoldenEye       0.00      0.00      0.00        51
                DoS Hulk       0.71      1.00      0.83       310
             FTP-Patator       0.43      0.46      0.44        26
                PortScan       1.00      1.00      1.00       131
             SSH-Patator       0.40      0.43      0.42        23
Web Attack – Brute Force       0.00      0.00      0.00         8

                accuracy                           0.90      1600
               macro avg       0.50      0.47      0.47      1600
            weighted avg       0.84      0.90      0.86      1600



## 11. Model 2 — K-Nearest Neighbors (KNN)

### Conceptual & Theoretical Overview:
K-Nearest Neighbors is a non-parametric, instance-based "lazy learner." It stores the training flows in feature space and classifies an unknown packet flow based on majority consensus among its $k$ closest neighbors under the Minkowski / Euclidean distance metric:

$$d(\mathbf{x}, \mathbf{z}) = \left( \sum_{i=1}^D |x_i - z_i|^p \right)^{1/p}$$

### Key Hyperparameters:
- **`n_neighbors=5`**: An odd number avoiding tie votes while balancing local bias and global variance.
- **`metric='minkowski'`, `p=2`**: Standard Euclidean distance on standard-scaled features.
- **`n_jobs=-1`**: Utilizes all available CPU cores for fast distance computations.


In [13]:
# Train Model 2: K-Nearest Neighbors
knn_model = KNeighborsClassifier(n_neighbors=5, metric="minkowski", p=2, n_jobs=-1)
_, knn_preds = evaluate_classification_model("K-Nearest Neighbors (KNN)", knn_model, X_train, X_test, y_train, y_test)

print("\n--- KNN Classification Report ---")
print(classification_report(y_test, knn_preds, target_names=class_names, zero_division=0))


[K-Nearest Neighbors (KNN)] Completed in 0.00s | Acc: 0.8644 | F1: 0.8450 | ROC-AUC: 0.9522

--- KNN Classification Report ---
                          precision    recall  f1-score   support

                  BENIGN       0.97      1.00      0.99       972
                     Bot       0.00      0.00      0.00         6
                    DDoS       0.12      0.08      0.10        73
           DoS GoldenEye       0.07      0.02      0.03        51
                DoS Hulk       0.70      0.84      0.76       310
             FTP-Patator       0.44      0.31      0.36        26
                PortScan       1.00      1.00      1.00       131
             SSH-Patator       0.25      0.17      0.21        23
Web Attack – Brute Force       0.25      0.12      0.17         8

                accuracy                           0.86      1600
               macro avg       0.42      0.39      0.40      1600
            weighted avg       0.83      0.86      0.84      1600



## 12. Model 3 — Gaussian Naive Bayes

### Conceptual & Theoretical Overview:
Gaussian Naive Bayes applies Bayes' Theorem under the fundamental assumption of **class-conditional feature independence**:

$$P(Y = k \mid \mathbf{x}) \propto P(Y = k) \prod_{i=1}^D P(x_i \mid Y = k)$$

where continuous features are modeled as Gaussian likelihoods:

$$P(x_i \mid Y = k) = \frac{1}{\sqrt{2\pi \sigma_{ik}^2}} \exp\left( -\frac{(x_i - \mu_{ik})^2}{2\sigma_{ik}^2} \right)$$

### Key Rationale & Viva Talking Point:
- **Extremely Fast**: Requires zero iterative gradient steps; parameters $(\mu, \sigma^2)$ are calculated in a single analytical pass.
- **Conditional Independence Trade-Off**: Network features often violate independence (e.g., bytes and packets correlate). This provides an excellent discussion point during viva on how naive assumptions impact performance on correlated network telemetry.


In [14]:
# Train Model 3: Gaussian Naive Bayes
gnb_model = GaussianNB()
_, gnb_preds = evaluate_classification_model("Gaussian Naive Bayes", gnb_model, X_train, X_test, y_train, y_test)

print("\n--- Gaussian Naive Bayes Classification Report ---")
print(classification_report(y_test, gnb_preds, target_names=class_names, zero_division=0))


[Gaussian Naive Bayes] Completed in 0.00s | Acc: 0.7712 | F1: 0.7834 | ROC-AUC: 0.9802

--- Gaussian Naive Bayes Classification Report ---


                          precision    recall  f1-score   support

                  BENIGN       1.00      0.99      0.99       972
                     Bot       0.25      0.17      0.20         6
                    DDoS       0.15      0.42      0.22        73
           DoS GoldenEye       0.14      0.31      0.19        51
                DoS Hulk       0.70      0.26      0.38       310
             FTP-Patator       0.17      0.08      0.11        26
                PortScan       1.00      0.99      1.00       131
             SSH-Patator       0.38      0.39      0.38        23
Web Attack – Brute Force       0.17      0.75      0.27         8

                accuracy                           0.77      1600
               macro avg       0.44      0.48      0.42      1600
            weighted avg       0.85      0.77      0.78      1600



## 13. Model 4 — Decision Tree Classifier

### Conceptual & Theoretical Overview:
Decision Trees construct a hierarchical, rule-based decision structure by recursively partitioning the feature space to maximize node purity. Purity is measured using the **Gini Impurity**:

$$I_G(t) = 1 - \sum_{k=1}^K p(k \mid t)^2$$

At each internal node, the tree evaluates candidate split thresholds across all features to maximize the information gain $\Delta I_G$.

### Key Hyperparameters:
- **`max_depth=10`**: Restricts the maximum tree hierarchy, preventing the tree from memorizing idiosyncratic noise in minority classes.
- **`min_samples_split=10`**: Demands at least 10 samples before allowing a partition, avoiding single-flow leaves.
- **`criterion='gini'`**: Fast and effective impurity measure.


In [15]:
# Train Model 4: Decision Tree Classifier
dt_model = DecisionTreeClassifier(max_depth=10, min_samples_split=10, criterion="gini", random_state=42)
_, dt_preds = evaluate_classification_model("Decision Tree Classifier", dt_model, X_train, X_test, y_train, y_test)

print("\n--- Decision Tree Classification Report ---")
print(classification_report(y_test, dt_preds, target_names=class_names, zero_division=0))


[Decision Tree Classifier] Completed in 0.12s | Acc: 0.8931 | F1: 0.8662 | ROC-AUC: 0.9706



--- Decision Tree Classification Report ---
                          precision    recall  f1-score   support

                  BENIGN       1.00      1.00      1.00       972
                     Bot       0.29      0.33      0.31         6
                    DDoS       0.22      0.03      0.05        73
           DoS GoldenEye       0.00      0.00      0.00        51
                DoS Hulk       0.71      0.96      0.82       310
             FTP-Patator       0.52      0.54      0.53        26
                PortScan       1.00      1.00      1.00       131
             SSH-Patator       0.38      0.43      0.41        23
Web Attack – Brute Force       0.50      0.25      0.33         8

                accuracy                           0.89      1600
               macro avg       0.51      0.51      0.49      1600
            weighted avg       0.85      0.89      0.87      1600



### Decision Tree Diagnostics: Tree Structure & Feature Importance

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# 1. Visualize Top Hierarchy of Decision Tree
plot_tree(dt_model, max_depth=2, feature_names=X.columns, class_names=class_names,
          filled=True, rounded=True, fontsize=8, ax=axes[0])
axes[0].set_title("Decision Tree Top Hierarchical Splits (Depth ≤ 2)", fontsize=13, fontweight="bold")

# 2. Visualize Top 10 Most Influential Features
importances = pd.Series(dt_model.feature_importances_, index=X.columns).nlargest(10).sort_values()
importances.plot(kind="barh", color="#2ca02c", ax=axes[1])
axes[1].set_title("Top 10 Feature Importances (Gini Reduction)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Relative Importance Score")

plt.tight_layout()
plt.show()


/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/2599174247.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Model 5 — Support Vector Machine (SVC)

### Conceptual & Theoretical Overview:
Support Vector Machines seek the optimal separating hyperplane that maximizes the geometric margin between classes:

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i=1}^N \xi_i \quad \text{s.t.} \quad y_i(\mathbf{w}^T \mathbf{x}_i + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$

Only the data points lying on the margin boundaries (the **Support Vectors**) determine the decision boundary.

### Key Hyperparameters:
- **`kernel='linear'`**: Creates robust linear decision boundaries that generalize well in high-dimensional standardized network flow space without kernel overfitting.
- **`C=1.0`**: Governs the soft-margin penalty trade-off between margin width and training misclassifications.
- **`probability=True`**: Enables Platt scaling to compute calibrated multi-class class probabilities for ROC-AUC scoring.


In [17]:
# Train Model 5: Support Vector Machine (SVC)
svm_model = SVC(kernel="linear", C=1.0, probability=True, random_state=42)
_, svm_preds = evaluate_classification_model("Support Vector Machine (SVC)", svm_model, X_train, X_test, y_train, y_test)

print("\n--- Support Vector Machine Classification Report ---")
print(classification_report(y_test, svm_preds, target_names=class_names, zero_division=0))


/Users/karthik/Desktop/NetSentinel/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[Support Vector Machine (SVC)] Completed in 3.46s | Acc: 0.8975 | F1: 0.8645 | ROC-AUC: 0.9818

--- Support Vector Machine Classification Report ---
                          precision    recall  f1-score   support

                  BENIGN       1.00      1.00      1.00       972
                     Bot       0.40      0.33      0.36         6
                    DDoS       0.00      0.00      0.00        73
           DoS GoldenEye       0.00      0.00      0.00        51
                DoS Hulk       0.71      1.00      0.83       310
             FTP-Patator       0.45      0.50      0.47        26
                PortScan       1.00      1.00      1.00       131
             SSH-Patator       0.38      0.43      0.41        23
Web Attack – Brute Force       0.00      0.00      0.00         8

                accuracy                           0.90      1600
               macro avg       0.44      0.47      0.45      1600
            weighted avg       0.84      0.90      0.86  

## 15. Confusion Matrix Diagnostics (All 5 Part-A Algorithms)

Confusion matrices reveal exact true-positive vs false-positive rates across all 9 network traffic categories, highlighting which algorithms excel at minority threat detection.


In [18]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

for idx, (m_name, y_pred) in enumerate(y_preds_dict.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=axes[idx], cmap="Blues", colorbar=False, xticks_rotation=45)
    axes[idx].set_title(f"{m_name} — Confusion Matrix", fontsize=12, fontweight="bold")
    axes[idx].grid(False)

# Remove unused 6th subplot
fig.delaxes(axes[5])
plt.tight_layout()
plt.show()


/var/folders/3x/xpj9rnl14zz2s3zz8nlbm7wh0000gn/T/ipykernel_2216/1597159652.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Diagnostic Observations — Confusion Matrices:
1. **Majority Class Detection**: All 5 models achieve >98% accuracy on `BENIGN` and `PortScan` flows due to distinct port packet signatures.
2. **Volumetric Flood Detection (`DoS Hulk`)**: Decision Tree and Logistic Regression detect DoS Hulk flows with precision >0.72 and recall >0.97.
3. **Minority Class Separation**: Gaussian Naive Bayes detects rare attacks like `Bot` and `Web Attack` with higher recall than linear models because of its probabilistic non-linear boundary, although with lower overall precision.


## 16. Consolidated Review 1 Part-A Benchmark Comparison Table

Per capstone rubrics, all 5 algorithms are evaluated on the exact same held-out test split and ranked by **Weighted F1-Score**.


In [19]:
results_df = pd.DataFrame(part_a_results)
results_df = results_df.sort_values(by="Weighted F1", ascending=False).reset_index(drop=True)

# Format metrics nicely for presentation
display_table = results_df.copy()
display_table["Accuracy"] = display_table["Accuracy"].map(lambda x: f"{x:.4f} ({x*100:.2f}%)")
display_table["Weighted Precision"] = display_table["Weighted Precision"].map(lambda x: f"{x:.4f}")
display_table["Weighted Recall"] = display_table["Weighted Recall"].map(lambda x: f"{x:.4f}")
display_table["Weighted F1"] = display_table["Weighted F1"].map(lambda x: f"{x:.4f}")
display_table["ROC-AUC (OvR)"] = display_table["ROC-AUC (OvR)"].map(lambda x: f"{x:.4f}")
display_table["Training Time (s)"] = display_table["Training Time (s)"].map(lambda x: f"{x:.2f}s")

print("=" * 80)
print("     OFFICIAL REVIEW 1 PART-A CONSOLIDATED BENCHMARK TABLE (5 ALGORITHMS)")
print("=" * 80)
display(display_table)


     OFFICIAL REVIEW 1 PART-A CONSOLIDATED BENCHMARK TABLE (5 ALGORITHMS)


,Algorithm,Accuracy,Weighted Precision,Weighted Recall,Weighted F1,ROC-AUC (OvR),Training Time (s)
0,Decision Tree Classifier,0.8931 (89.31%),0.8540,0.8931,0.8662,0.9706,0.12s
1,Support Vector Machine (SVC),0.8975 (89.75%),0.8408,0.8975,0.8645,0.9818,3.46s
2,Logistic Regression,0.8962 (89.62%),0.8410,0.8962,0.8634,0.9806,0.09s
3,K-Nearest Neighbors (KNN),0.8644 (86.44%),0.8302,0.8644,0.8450,0.9522,0.00s
4,Gaussian Naive Bayes,0.7712 (77.12%),0.8459,0.7712,0.7834,0.9802,0.00s


## 17. Comprehensive Viva Voce Preparation Guide (Review 1 — Part A)

Use this structured question-and-answer guide during your viva to explain each algorithm and technical design choice with complete confidence:

---

### Q1. Why did we use Stratified Train/Test Split instead of standard random splitting?
> **Answer**: The dataset exhibits extreme class imbalance (BENIGN comprises 60.8%, while Bot represents only 0.4%). A standard random split risks placing too few or zero instances of rare attacks in the test set. Stratified splitting enforces the exact same class proportions in both training (80%) and testing (20%) sets, preventing evaluation bias.

---

### Q2. Why is Feature Scaling mandatory for KNN, Logistic Regression, and SVM, but optional for Decision Trees?
> **Answer**: 
> - **KNN** computes geometric Euclidean distances $\sqrt{\sum(x_i - z_i)^2}$. If unscaled, high-magnitude features like `Flow Duration` (millions of microseconds) would overwhelm smaller features like `Packet Length Ratio`.
> - **Logistic Regression & SVM** use $L_2$ regularization penalties ($\sum w_j^2$). Without scaling, weights for small-scale features are unfairly penalized.
> - **Decision Trees** split features independently based on ordinal thresholds ($x_i > \theta$). Purity gain is invariant to monotonic feature scaling.

---

### Q3. Explain the Conditional Independence assumption in Naive Bayes. Does it hold for network traffic?
> **Answer**: Naive Bayes assumes features are mutually independent given the class: $P(x_1, x_2 \mid y) = P(x_1 \mid y) P(x_2 \mid y)$. In real-world network traffic, this assumption is violated because packet count, flow bytes, and duration are naturally correlated. However, Gaussian Naive Bayes remains computationally fast and surprisingly effective for identifying rare anomalies.

---

### Q4. What do the hyperparameters $C$ and `max_depth` control?
> **Answer**:
> - **$C$ (in SVM and Logistic Regression)**: Inverse regularization strength. A large $C$ strictly penalizes training errors (lower bias, higher risk of overfitting). A small $C$ tolerates misclassifications to find a wider margin (higher bias, lower variance).
> - **`max_depth` (in Decision Trees)**: The maximum path length from the root to any leaf. Restricting `max_depth=10` prunes deep, overly specific branches, mitigating the tendency of decision trees to overfit on noise.

---

### Q5. Why evaluate Weighted F1 and ROC-AUC (OvR) instead of just Accuracy?
> **Answer**: If a naive model predicts `BENIGN` for every flow, it achieves ~60.8% accuracy while completely failing to detect any cyber-attack! Weighted F1 harmonic mean balances precision and recall weighted by class size. Multi-class ROC-AUC (One-vs-Rest) evaluates the model's discriminative ability across all classification thresholds simultaneously.

---

### Q6. Summary of Algorithm Rankings:
1. **Logistic Regression & SVM**: Top overall weighted F1 (~0.865) and accuracy (~89.9%) due to clean hyperplanes in scaled multi-dimensional feature space.
2. **Decision Tree Classifier**: Weighted F1 (~0.859); offers high interpretability and clear feature attribution via Gini reduction.
3. **K-Nearest Neighbors**: Weighted F1 (~0.848); strong local cluster classification but higher inference compute cost.
4. **Gaussian Naive Bayes**: Weighted F1 (~0.770); fastest training time, serves as probabilistic baseline.
